# 02 — Results & Evaluation
**ML-Based 6G Channel Modeling and Beam Prediction | VIT Chennai | BECE311L**

Loads the best checkpoint, evaluates all PRD acceptance criteria, and generates
the three required result plots.

| Metric | Target | Description |
|--------|--------|-------------|
| Top-1 Accuracy | ≥ 85% | Exact best-beam prediction |
| Top-5 Accuracy | ≥ 97% | True beam in top-5 predictions |
| Rate Ratio | ≥ 0.95 | Mean predicted rate / mean optimal rate @ 20 dB SNR |

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt

plt.style.use('dark_background')
matplotlib.rcParams.update({
    'font.family':    'DejaVu Sans',
    'axes.facecolor': '#0f1117',
    'figure.facecolor':'#0f1117',
    'axes.edgecolor': '#444',
    'grid.color':     '#2a2d3e',
    'grid.linestyle': '--',
    'grid.alpha':     0.5,
})

from src.dataset import (
    dft_codebook, load_deepmimo_v4,
    filter_valid_users, compute_beam_labels, make_dataloaders,
)
from src.model import BeamPredictor
from src.evaluate import (
    topk_accuracy, evaluate_rates, evaluate_all,
    plot_training_curve, plot_beam_map, plot_rate_cdf,
)

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINT  = '../checkpoints/best_mlp.pt'
HISTORY     = '../checkpoints/training_history.json'
RESULTS_DIR = '../results'
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Device     : {DEVICE}')
print(f'Checkpoint : {CHECKPOINT}')
print(f'Exists     : {os.path.exists(CHECKPOINT)}')

## 1. Load Real Dataset — asu_campus_3p5

In [ ]:
positions, channels = load_deepmimo_v4('asu_campus_3p5', N_t=64)
channels, positions = filter_valid_users(channels, positions)

codebook = dft_codebook(N_t=64, N_b=64)
labels   = compute_beam_labels(channels, codebook)

train_loader, val_loader, test_loader, scaler, ch_split = make_dataloaders(
    positions, labels, channels, batch_size=256
)
channels_test = ch_split['test']
print(f'Test users: {len(channels_test):,}')

## 2. Load Checkpoint

In [ ]:
assert os.path.exists(CHECKPOINT), f'Run: python src/train.py   (from repo root)'

ckpt  = torch.load(CHECKPOINT, map_location=DEVICE)
cfg   = ckpt.get('config', {})
model = BeamPredictor(
    input_dim   = cfg.get('input_dim',    3),
    hidden_dims = cfg.get('hidden_dims',  [512, 256, 128]),
    num_classes = cfg.get('num_classes',  64),
    dropout     = cfg.get('dropout',      0.2),
    n_res_blocks= cfg.get('n_res_blocks', 2),
).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

print(f'Loaded epoch {ckpt["epoch"]}  |  val_acc = {ckpt["val_acc"]*100:.2f}%')
print(f'Architecture: {cfg.get("hidden_dims")} | '
      f'{cfg.get("n_res_blocks",2)} res-blocks | '
      f'dropout={cfg.get("dropout",0.2)}')

## 3. Plot 1 — Training Curve

In [ ]:
with open(HISTORY) as f:
    history = json.load(f)

epochs     = np.arange(1, len(history['train_loss']) + 1)
train_loss = np.array(history['train_loss'])
val_acc    = np.array(history['val_acc']) * 100.0
best_epoch = history.get('best_epoch', 0)

fig, ax1 = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor('#0f1117')
ax1.set_facecolor('#0f1117')

C_LOSS, C_ACC = '#7c83fd', '#f7b731'

ln1 = ax1.plot(epochs, train_loss, color=C_LOSS, lw=1.8,
               label='Train Loss', alpha=0.9)
ax1.set_xlabel('Epoch', fontsize=13, color='white')
ax1.set_ylabel('Cross-Entropy Loss', fontsize=13, color=C_LOSS)
ax1.tick_params(axis='y', labelcolor=C_LOSS, colors='white')
ax1.tick_params(axis='x', colors='white')
for spine in ax1.spines.values(): spine.set_edgecolor('#444')

ax2 = ax1.twinx()
ax2.set_facecolor('#0f1117')
ln2 = ax2.plot(epochs, val_acc, color=C_ACC, lw=1.8,
               label='Val Top-1 Acc', alpha=0.9)
ax2.set_ylabel('Top-1 Accuracy (%)', fontsize=13, color=C_ACC)
ax2.tick_params(axis='y', labelcolor=C_ACC, colors='white')
ax2.axhline(85, color='#ff6b6b', ls='--', lw=1.5, label='85% target', alpha=0.8)
if best_epoch:
    ax2.axvline(best_epoch, color='#26de81', ls=':', lw=1.5,
                label=f'Best (ep {best_epoch}: {history["best_val_acc"]*100:.1f}%)')

lines  = ln1 + ln2
labels_legend = [l.get_label() for l in lines]
ax1.legend(lines, labels_legend, loc='lower right',
           facecolor='#1a1d2e', edgecolor='#444', labelcolor='white', fontsize=10)
ax2.legend(loc='upper right', facecolor='#1a1d2e', edgecolor='#444',
           labelcolor='white', fontsize=10)

plt.title('Training Curve — 6G Beam Prediction (Deep Residual MLP)',
          fontsize=14, color='white', pad=12)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/training_curve.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print(f'Best val accuracy : {history["best_val_acc"]*100:.2f}% (epoch {best_epoch})')

## 4. Full Evaluation — All Metrics

In [ ]:
metrics = evaluate_all(
    model, test_loader, channels_test, codebook,
    snr_db=20.0, device=DEVICE
)

## 5. Plot 2 — Spatial Beam Map (Predicted vs Optimal)

In [ ]:
N_test   = len(metrics['pred_beams'])
pos_test = positions[-N_test:]

fig, axes = plt.subplots(1, 2, figsize=(17, 7))
fig.patch.set_facecolor('#0f1117')

for ax, beam_arr, title in [
    (axes[0], metrics['optimal_beams'], 'Ground-Truth Optimal Beams'),
    (axes[1], metrics['pred_beams'],    'Model Predicted Beams'),
]:
    ax.set_facecolor('#0f1117')
    sc = ax.scatter(pos_test[:,0], pos_test[:,1],
                    c=beam_arr, cmap='hsv', vmin=0, vmax=63,
                    s=5, alpha=0.75, linewidths=0)
    cbar = fig.colorbar(sc, ax=ax, fraction=0.04, pad=0.02)
    cbar.set_label('Beam Index', fontsize=11, color='white')
    cbar.ax.yaxis.set_tick_params(color='white')
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')
    ax.set_xlabel('X (m)', fontsize=12, color='white')
    ax.set_ylabel('Y (m)', fontsize=12, color='white')
    ax.set_title(title, fontsize=13, color='white', pad=10)
    ax.tick_params(colors='white')
    ax.grid(True, alpha=0.3)

plt.suptitle('Spatial Beam Map — asu_campus_3p5 Test Set',
             fontsize=15, color='white', y=1.01)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/beam_map.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## 6. Plot 3 — Achievable Rate CDF @ 20 dB SNR

In [ ]:
def _cdf(data):
    x = np.sort(data)
    y = np.arange(1, len(x)+1) / len(x)
    return x, y

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#0f1117')

for rates, label, color, lw in [
    (metrics['rates_opt'],  'Optimal Beam',   '#26de81', 2.2),
    (metrics['rates_pred'], 'Predicted Beam', '#f7b731', 2.2),
    (metrics['rates_rand'], 'Random Beam',    '#ff6b6b', 1.5),
]:
    x, y = _cdf(rates)
    ax.plot(x, y*100, label=label, color=color, lw=lw)

# Shade the gap between predicted and optimal
x_opt,  y_opt  = _cdf(metrics['rates_opt'])
x_pred, y_pred = _cdf(metrics['rates_pred'])

ax.set_xlabel('Achievable Rate (bps/Hz)', fontsize=13, color='white')
ax.set_ylabel('CDF (%)', fontsize=13, color='white')
ax.set_title(f'Achievable Rate CDF @ SNR = 20 dB  |  '
             f'Rate Ratio = {metrics["rate_ratio"]:.3f}',
             fontsize=13, color='white', pad=12)
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_edgecolor('#444')
ax.legend(facecolor='#1a1d2e', edgecolor='#444', labelcolor='white', fontsize=11)
ax.grid(True)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/rate_cdf.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## 7. Top-5 Accuracy vs SNR Sweep

In [ ]:
from src.evaluate import achievable_rate_batch

snr_range  = np.arange(-10, 31, 5)
rate_ratios = []

for snr in snr_range:
    r_opt  = achievable_rate_batch(channels_test, metrics['optimal_beams'],  codebook, snr)
    r_pred = achievable_rate_batch(channels_test, metrics['pred_beams'],     codebook, snr)
    rate_ratios.append(r_pred.mean() / max(r_opt.mean(), 1e-10))

fig, ax = plt.subplots(figsize=(9, 4))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#0f1117')
ax.plot(snr_range, np.array(rate_ratios)*100, color='#7c83fd', lw=2.2,
        marker='o', ms=5, label='Rate Ratio (%)')
ax.axhline(95, color='#ff6b6b', ls='--', lw=1.5, label='95% target')
ax.set_xlabel('SNR (dB)', fontsize=13, color='white')
ax.set_ylabel('Rate Ratio (%)', fontsize=13, color='white')
ax.set_title('Achievable Rate Ratio vs SNR', fontsize=14, color='white')
ax.set_ylim(0, 105)
ax.tick_params(colors='white')
ax.legend(facecolor='#1a1d2e', edgecolor='#444', labelcolor='white')
ax.grid(True)
for spine in ax.spines.values(): spine.set_edgecolor('#444')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/rate_ratio_vs_snr.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

## 8. Final Results Summary

In [ ]:
top1_pass  = metrics['top1_acc']  >= 0.85
top5_pass  = metrics['top5_acc']  >= 0.97
ratio_pass = metrics['rate_ratio'] >= 0.95
all_pass   = top1_pass and top5_pass and ratio_pass

print('═' * 62)
print('  FINAL EVALUATION SUMMARY')
print('  6G Beam Prediction | VIT Chennai | BECE311L | May 2026')
print('═' * 62)
print(f'  Dataset   : asu_campus_3p5 (outdoor 3.5 GHz, 131,931 UEs)')
print(f'  Model     : Deep Residual MLP ([512]×ResBlock×2→[256,128]→64)')
print(f'  Params    : 1,231,040')
print(f'  Epochs    : {len(history["train_loss"])}')
print()
print(f'  {"Metric":<22} {"Value":>10}   {"Target":>8}   {"Status"}')
print(f'  {"─"*58}')
print(f'  {"Top-1 Accuracy":<22} {metrics["top1_acc"]*100:>9.2f}%   {"≥ 85%":>8}   {"✓ PASS" if top1_pass else "✗ FAIL"}')
print(f'  {"Top-5 Accuracy":<22} {metrics["top5_acc"]*100:>9.2f}%   {"≥ 97%":>8}   {"✓ PASS" if top5_pass else "✗ FAIL"}')
print(f'  {"Rate (Predicted)":<22} {metrics["mean_pred"]:>9.4f}    {"bps/Hz":>8}')
print(f'  {"Rate (Optimal)":<22} {metrics["mean_opt"]:>9.4f}    {"bps/Hz":>8}')
print(f'  {"Rate (Random)":<22} {metrics["mean_rand"]:>9.4f}    {"bps/Hz":>8}')
print(f'  {"Rate Ratio":<22} {metrics["rate_ratio"]:>10.4f}   {"≥ 0.95":>8}   {"✓ PASS" if ratio_pass else "✗ FAIL"}')
print()
print(f'  Overall: {"✓ ALL CRITERIA MET" if all_pass else "✗ Some criteria not met"}')
print('═' * 62)